## التقييم المقارن الشامل للنظام

هذا الـ Notebook هو الأداة النهائية لتقييم النظام. يقوم بالمهام التالية:
1.  **مرونة اختيار مجموعة البيانات:** يمكنك بسهولة تغيير مجموعة البيانات المستهدفة (`antique` أو `beir/quora`).
2.  **ضبط المعاملات:** يقوم تلقائياً بضبط معاملات BM25.
3.  **التقييم المقارن:** يقوم بتقييم النظام مرتين:
    - **مرة قبل** تفعيل ميزة إعادة الترتيب بـ NER (الطلبات الأساسية فقط).
    - **مرة بعد** تفعيل ميزة NER (مع الطلب الإضافي).
4.  **عرض شامل للنتائج:** يعرض النتائج في جداول مقارنة واضحة.

### الخطوة 1: الإعدادات العامة

In [1]:
import requests
import ir_datasets
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- الإعدادات التي يمكنك تغييرها ---
DATASET_TO_EVALUATE = 'antique' # غيّر هذه القيمة إلى 'beir/quora' للتقييم على المجموعة الأخرى
QUERY_SAMPLE_SIZE = None # اجعلها None للتقييم الكامل والنهائي
# -----------------------------------

project_root = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(project_root)
from config import API_PORTS

SEARCH_API_URL = f"http://127.0.0.1:{API_PORTS['SEARCH']}/search/"
MODELS_TO_EVALUATE = ['tfidf', 'bm25', 'bert', 'hybrid']
IR_DATASET_NAME = f'{DATASET_TO_EVALUATE}/test' if DATASET_TO_EVALUATE == 'antique' else DATASET_TO_EVALUATE

### الخطوة 2: تحميل البيانات وتعريف الدوال

In [2]:
# تحميل مجموعة البيانات
print(f"Loading dataset: {IR_DATASET_NAME}...")
dataset = ir_datasets.load(IR_DATASET_NAME)
queries = {q.query_id: q.text for q in dataset.queries_iter()}
# تعديل بسيط هنا لضمان عدم وجود أخطاء في qrels
qrels = {}
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0:
        if qrel.query_id not in qrels:
            qrels[qrel.query_id] = set()
        qrels[qrel.query_id].add(qrel.doc_id)


if QUERY_SAMPLE_SIZE:
    # التأكد من أننا نأخذ فقط الـ IDs التي لها إجابات صحيحة في qrels
    query_ids_with_rels = [qid for qid in queries.keys() if qid in qrels]
    query_ids_to_run = query_ids_with_rels[:QUERY_SAMPLE_SIZE]
else:
    query_ids_to_run = [qid for qid in queries.keys() if qid in qrels]

print(f"Loaded data. Running evaluation on {len(query_ids_to_run)} queries that have relevance judgments.")

# --- تعريف دوال التقييم (مع التصحيح) ---
def calculate_metrics(retrieved_docs, relevant_docs):
    """Calculates a dictionary of metrics for a single query result."""
    if not retrieved_docs or not relevant_docs:
        return {'AP': 0.0, 'RR': 0.0, 'P@10': 0.0, 'R@10': 0.0}
    
    hits = 0
    sum_of_precisions = 0.0
    reciprocal_rank = 0.0
    first_hit_found = False
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id in relevant_docs:
            if not first_hit_found:
                reciprocal_rank = 1 / (i + 1)
                first_hit_found = True
            hits += 1
            sum_of_precisions += hits / (i + 1)
    
    average_precision = sum_of_precisions / len(relevant_docs)
    
    hits_at_10 = len(set(retrieved_docs[:10]).intersection(relevant_docs))
    precision_at_10 = hits_at_10 / 10.0
    recall_at_10 = hits_at_10 / len(relevant_docs)
    
    return {
        'AP': average_precision,
        'RR': reciprocal_rank,
        'P@10': precision_at_10,
        'R@10': recall_at_10
    }

def search_system(query_text, model_type, k1, b, ner, weight):
    """Calls the Search API."""
    payload = {
        "dataset_name": DATASET_TO_EVALUATE, "query": query_text, "model_type": model_type,
        "top_k": 100, "k1": k1, "b": b, "enable_ner_reranking": ner, "hybrid_bm25_weight": weight
    }
    try:
        r = requests.post(SEARCH_API_URL, json=payload, timeout=60)
        r.raise_for_status()
        return [res['doc_id'] for res in r.json().get('results', [])]
    except requests.exceptions.RequestException:
        return []

# --- الدالة التي تم تصحيحها ---
def evaluate_model(query_ids_list, model_type, k1=1.5, b=0.75, ner=False, weight=0.8):
    """
    Evaluates a model on a given list of query IDs.
    It now correctly uses the global 'queries' and 'qrels' dictionaries.
    """
    all_metrics = []
    # الآن الدالة تتكرر بشكل صحيح على قائمة المعرفات
    for query_id in query_ids_list:
        # وتستخدم القواميس العامة لجلب البيانات
        query_text = queries.get(query_id)
        relevant_docs = qrels.get(query_id, set())
        
        if not query_text or not relevant_docs:
            continue
            
        retrieved_docs = search_system(query_text, model_type, k1, b, ner, weight)
        all_metrics.append(calculate_metrics(retrieved_docs, relevant_docs))
        
    return pd.DataFrame(all_metrics).mean().to_dict() if all_metrics else {}


Loading dataset: antique/test...
Loaded data. Running evaluation on 200 queries that have relevance judgments.


### الخطوة 3: ضبط معاملات BM25 (يُنفذ مرة واحدة)

In [3]:
k1_vals = [1.2, 1.6, 2.0]; b_vals = [0.65, 0.75, 0.85]
tuning_res = []
pbar = tqdm(total=len(k1_vals) * len(b_vals), desc="Tuning BM25")
for k1 in k1_vals:
    for b in b_vals:
        metrics = evaluate_model(query_ids_to_run, 'bm25', k1, b, False, 0.8)
        tuning_res.append({'k1': k1, 'b': b, 'MAP': metrics.get('AP', 0)})
        pbar.update(1)
pbar.close()
tuning_df = pd.DataFrame(tuning_res)
best_params = tuning_df.loc[tuning_df['MAP'].idxmax()]
best_k1, best_b = best_params['k1'], best_params['b']
print(f"Best Parameters Found: k1={best_k1}, b={best_b} with MAP={best_params['MAP']:.4f}")

Tuning BM25: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [09:24<00:00, 62.77s/it]

Best Parameters Found: k1=1.2, b=0.65 with MAP=0.4239


### الخطوة 4: التقييم المقارن (قبل وبعد تفعيل NER)

In [4]:
print("--- Running Comparative Evaluation ---")
final_results_list = []

# --- الجزء الأول: حلقة التقييم (التي كانت مفقودة) ---
# سيتم الآن تشغيل التقييم لكل سيناريو (مع وبدون NER)
for ner_status in [False, True]:
    scenario_name = "With NER Re-ranking" if ner_status else "Baseline (No NER)"
    print(f"\nEvaluating scenario: {scenario_name}")
    
    # التأكد من أن MODELS_TO_EVALUATE و query_ids_to_run معرفة من الخلايا السابقة
    if 'MODELS_TO_EVALUATE' not in locals() or 'query_ids_to_run' not in locals():
        raise NameError("Please run the previous setup cells first.")

    for model_type in tqdm(MODELS_TO_EVALUATE, desc=scenario_name):
        # استخدام أفضل معاملات BM25 التي وجدناها
        metrics = evaluate_model(query_ids_to_run, model_type, best_k1, best_b, ner_status, 0.8)
        
        metrics['Model'] = model_type
        metrics['Scenario'] = scenario_name
        final_results_list.append(metrics)

# --- الجزء الثاني: معالجة وعرض النتائج (الذي كان يسبب الخطأ) ---
print("\n--- Processing and Displaying Final Results ---")

# الآن المتغير final_results_list موجود بالتأكيد
final_df = pd.DataFrame(final_results_list)

# إعادة تسمية الأعمدة لتوحيد الأسماء
if 'AP' in final_df.columns:
    final_df.rename(columns={'AP': 'MAP'}, inplace=True)
if 'RR' in final_df.columns:
    final_df.rename(columns={'RR': 'MRR'}, inplace=True)

# فصل النتائج لعرضها في جدولين منفصلين
baseline_df = final_df[final_df['Scenario'] == 'Baseline (No NER)'].set_index('Model')[['MAP', 'MRR', 'P@10', 'R@10']]
ner_df = final_df[final_df['Scenario'] == 'With NER Re-ranking'].set_index('Model')[['MAP', 'MRR', 'P@10', 'R@10']]

print(f"\n--- Baseline Evaluation Results (NER Disabled) ---")
display(baseline_df.style.format('{:.4f}')\
                      .background_gradient(cmap='viridis_r', axis=0)\
                      .set_caption("مقارنة أداء النماذج (بدون تحسين NER)"))

print(f"\n--- Enhanced Evaluation Results (NER Enabled) ---")
display(ner_df.style.format('{:.4f}')\
                      .background_gradient(cmap='viridis', axis=0)\
                      .set_caption("مقارنة أداء النماذج (مع تحسين NER)"))


--- Running Comparative Evaluation ---

Evaluating scenario: Baseline (No NER)


Baseline (No NER): 100%|███████████████████████████████████████████████████████████████████████████████████████| 4/4 [02:28<00:00, 37.20s/it]



Evaluating scenario: With NER Re-ranking


With NER Re-ranking: 100%|█████████████████████████████████████████████████████████████████████████████████████| 4/4 [06:24<00:00, 96.05s/it]



--- Processing and Displaying Final Results ---

--- Baseline Evaluation Results (NER Disabled) ---


,MAP,MRR,P@10,R@10
Model,,,,
tfidf,0.1867,0.7240,0.3660,0.1205
bm25,0.4239,0.9304,0.7330,0.2392
bert,0.2043,0.8387,0.4515,0.1479
hybrid,0.4520,0.9421,0.7595,0.2495



--- Enhanced Evaluation Results (NER Enabled) ---


,MAP,MRR,P@10,R@10
Model,,,,
tfidf,0.1658,0.7238,0.3660,0.1205
bm25,0.3913,0.9304,0.7330,0.2392
bert,0.1900,0.8386,0.4515,0.1479
hybrid,0.4142,0.9401,0.7570,0.2485
